In [1026]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import spikegen
import numpy as np
import pandas as pd
import os
from torch.utils.data import TensorDataset, DataLoader

In [1027]:
def extract_and_normalize_top_features(df, top_features):
    """
        df: combined_data DataFrame
        top_features: list of top N features

        returns a normalized tensor
    """

    X = df[top_features].values
    X = torch.tensor(X, dtype=torch.float32)

    # Normalize features to range [0, 1]
    X_min = X.min(dim=0).values
    X_max = X.max(dim=0).values
    X_normalized = (X - X_min) / (X_max - X_min + 1e-8)  # avoid division by zero

    '''
    # safety cleanup
    X_norm = torch.nan_to_num(X_norm, nan=0.0, posinf=1.0, neginf=0.0)
    X_norm = X_norm.clamp(0, 1)
    '''
    return X_normalized

In [1028]:
def load_data(fold_num):
    return pd.read_csv(f"bcg_dataset/feat_fold_{fold_num}.csv")

In [1029]:
def load_all_data_with_fold_indices():
    '''
    Load and combine all pre-split folds into one dataset
    
    The original dataset comes pre-split into 5 folds, but I combine them so I can split using
    groups over the patient id
    '''
    all_folds = []
    fold_indices = {}
    current_idx = 0

    for fold_num in range(5):
        fold_data = load_data(fold_num)
        n = len(fold_data)
        fold_indices[fold_num] = list(range(current_idx, current_idx + n))
        current_idx += n
        all_folds.append(fold_data)
    
    combined = pd.concat(all_folds, ignore_index=True)
    print(f'Combined dataset shape: {combined.shape}')
    print(f'Total samples: {len(combined)}')
    for f, idx in fold_indices.items():
        print(f'Fold {f}: {len(idx)} samples mapped to indices [{idx[0]}..{idx[-1]}]')
    
    return combined, fold_indices

In [1030]:
def naive_baseline_cv(Y_SBP, Y_DBP, fold_indices, num_folds):
    """
    Cross-validated naive baseline:
    Predict the mean of the training fold.
    """

    results = {
        'SBP': {'maes': [], 'rmses': []},
        'DBP': {'maes': [], 'rmses': []}
    }

    for fold in range(num_folds):

        test_idx = fold_indices[fold]
        train_idx = []
        for f in range(num_folds):
            if f != fold:
                train_idx.extend(fold_indices[f])
    
        y_sbp_train = Y_SBP[train_idx]
        y_sbp_test  = Y_SBP[test_idx]

        y_dbp_train = Y_DBP[train_idx]
        y_dbp_test  = Y_DBP[test_idx]

        mean_sbp = y_sbp_train.mean()
        mean_dbp = y_dbp_train.mean()

        #Constant prediction
        pred_sbp = torch.full_like(y_sbp_test, mean_sbp)
        pred_dbp = torch.full_like(y_dbp_test, mean_dbp)

        mae_sbp  = (pred_sbp - y_sbp_test).abs().mean().item()
        rmse_sbp = ((pred_sbp - y_sbp_test)**2).mean().sqrt().item()

        mae_dbp  = (pred_dbp - y_dbp_test).abs().mean().item()
        rmse_dbp = ((pred_dbp - y_dbp_test)**2).mean().sqrt().item()

        results['SBP']['maes'].append(mae_sbp)
        results['SBP']['rmses'].append(rmse_sbp)
        results['DBP']['maes'].append(mae_dbp)
        results['DBP']['rmses'].append(rmse_dbp)

    print("="*50)
    print("NAIVE BASELINE RESULTS")
    print("="*50)

    for tag in ['SBP', 'DBP']:
        maes  = results[tag]['maes']
        rmses = results[tag]['rmses']
        print(f"{tag}: "
              f"MAE {np.mean(maes):.4f} ± {np.std(maes):.4f} | "
              f"RMSE {np.mean(rmses):.4f}")

    print("="*50)

    return results

In [1031]:
"""
Params:
    sample_tensor : torch.Tensor of shape [50, 30]
        Spike train for ONE sample:
        50 time steps * 30 input features

Returns:
    spike_times : list of length 30
        Each element is a list of spike times (ms) for one feature neuron.
        This format is required for NEST spike_generator nodes.

Note:
    The full dataset has shape [50, 3063, 30].
    You must select one sample first:
        sample_tensor = spikes_SBP[:, sample_idx, :]
"""

def tensor_to_spike_times(sample_tensor, dt=1.0):
    spike_times = []
    
    # shape[1] = 30
    for f in range(sample_tensor.shape[1]):
        t_idx = torch.where(sample_tensor[:,f] > 0)[0] # only extract the spike times where spikes for feature f occured
        times = ((t_idx.numpy() + 1) * dt).astype(float).tolist() # shift by 1 to avoid spike times of 0.0
        spike_times.append(times)

    return spike_times

In [1032]:
import torch
import torch.nn as nn
import torch.optim as optim

class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [1033]:
# def build_snn(n_input, n_hidden, n_output, beta=0.9, threshold=0.9):
class SNN(nn.Module):
    def __init__(self, n_input, n_hidden, n_output, beta, threshold):
        super().__init__()
        self.fc1 = nn.Linear(n_input,n_hidden, bias=True)
        self.lif1 = snn.Leaky(beta=beta, threshold=threshold)
        self.fc2 = nn.Linear(n_hidden,n_output, bias=False)
        self.lif2 = snn.Leaky(beta=beta, threshold=threshold)
    def forward(self, spike_input):
        '''
        spike_input: [timesteps, n_features]
        Returns:
            spk_out_rec = [] 
            mem_out_rec = [] 
        '''
        # reset membrane potentials for both layer
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk_out_rec = []    # collect output spikes at each timestep
        mem_out_rec = []    # collect membrane potentials at each timestep
        for step in range(spike_input.shape[0]):
            cur1 = self.fc1(spike_input[step])       # n_hidden
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)                 # n_output
            spk2, mem2 = self.lif2(cur2, mem2)

            spk_out_rec.append(spk2)
            mem_out_rec.append(mem2)

        return torch.stack(spk_out_rec), torch.stack(mem_out_rec)

In [1034]:
def decode(spk_out, value_range):
    """
    spk_out    : [T, batch, n_output]
    value_range: [n_output] tensor
    Returns    : [batch] predicted BP values
    """
    
    firing_rates = spk_out.mean(dim=0)
    denom        = firing_rates.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return (firing_rates * value_range).sum(dim=1) / denom.squeeze(1)

In [1035]:
def get_fold_loaders(fold, num_folds, spikes, Y_SBP, Y_DBP, fold_indices, batch_size):
    '''
    Params: 
        - fold: current fold
        - num_folds: number of total folds
        - spikes: spike trains, shape [50, 3063, 30]
        - Y_SBP: all SBP labels
        - Y_DBP: all DBP labels 
        - fold_indices: 
        - batch_size: samples per batch
    Returns: 
        - Returns the variable train_loader which consists of 
    Note: 
        - 
    '''
    test_data = load_data(fold)
    train_data = pd.concat([load_data(i) for i in range(num_folds) if i!= fold])
    test_idx = fold_indices[fold]

    train_idx = []
    for f in range(num_folds):
        if f != fold:
            train_idx.extend(fold_indices[f])
    
    X = spikes.permute(1,0,2)
    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = torch.stack([Y_SBP[train_idx], Y_DBP[train_idx]], dim=1)
    y_test = torch.stack([Y_SBP[test_idx], Y_DBP[test_idx]], dim=1)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=batch_size, shuffle=False
    )

    return train_loader, test_loader

In [1036]:
def eval_fold(model, test_loader, value_range, target_col):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.permute(1,0,2)
            spk_out, _ = model(batch_x)
            pred = decode(spk_out,value_range)
            all_preds.append(pred)
            all_labels.append(batch_y[:, target_col])

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    mae = (all_preds - all_labels).abs().mean().item()
    rmse = ((all_preds - all_labels)**2).mean().sqrt().item()

    return mae, rmse

In [1037]:
def train_fold(model_sbp, model_dbp, train_loader, range_SBP, range_DBP, num_epochs, lr):

        # train the fold
        optimizier_sbp = torch.optim.Adam(model_sbp.parameters(), lr=lr)
        optimizier_dbp = torch.optim.Adam(model_dbp.parameters(), lr=lr)

        loss_fn = nn.MSELoss()

        for epoch in range(num_epochs):
            model_sbp.train()
            model_dbp.train()
            total_loss_sbp, total_loss_dbp, num_batches = 0,0,0
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.permute(1,0,2)
                
                optimizier_sbp.zero_grad()
                spk_out, _ = model_sbp(batch_x)
                pred = decode(spk_out, range_SBP)
                loss_sbp = loss_fn(pred, batch_y[:,0])
                loss_sbp.backward()
                optimizier_sbp.step()
                
                optimizier_dbp.zero_grad()
                spk_out, _ = model_dbp(batch_x)
                pred = decode(spk_out, range_DBP)
                loss_dbp = loss_fn(pred, batch_y[:,1])
                loss_dbp.backward()
                optimizier_dbp.step()

                total_loss_sbp += loss_sbp.item()
                total_loss_dbp += loss_dbp.item()
                num_batches += 1
        
        print(f'Epoch {epoch+1:>2}/{num_epochs}')
        print(f'SBP MSE: {loss_sbp/num_batches:.4f} | DBP MSE: {loss_dbp/num_batches:.4f}')

In [1038]:
def run_experiment(spikes, fold_indices, Y_SBP, Y_DBP, range_SBP, range_DBP, config, experiment_name):
    '''
    Params:
        - spikes: Spike trains containing 0 or 1 which were generated by FeatureEncoding.ipynb, shape = [50, 3063, 30]
        - labels: Value to predict (either SP or DP)
        - value_range: Predefined value ranges for SBP and DBP
        - config: The configuration of our model
        - experiment_name: Name of the experiment, i.e. which encoding scheme we are using and which value we are predicting
    Returns:
        - fold_maes and fold_rmses errors that were obtained per fold training
        - mean_mae and mean_rmse error which are the mean fold errors across all folds
    Note:
        - 
    '''
    print(f'Running {experiment_name}')

    results = {'SBP': {'maes': [], 'rmses': []}, 
               'DBP': {'maes': [], 'rmses': []}           
    }

    fold_maes, fold_rmses = [], []
    
    for fold in range(config['num_folds']):
        print(f'running fold {fold+1}/{config['num_folds']}')
        model_sbp = SNN(
            n_input=config['n_input'],
            n_hidden=config['n_hidden'],
            n_output=config['n_output'],
            beta=config['beta'],
            threshold=config['threshold']
        )
        model_dbp = SNN(
            n_input=config['n_input'],
            n_hidden=config['n_hidden'],
            n_output=config['n_output'],
            beta=config['beta'],
            threshold=config['threshold']
        )

        train_loader, test_loader = get_fold_loaders(fold, config['num_folds'], spikes, Y_SBP, Y_DBP, fold_indices, config['batch_size'])

        train_fold(model_sbp, model_dbp, train_loader, range_SBP, range_DBP, config['num_epochs'], config['lr'])


        mae_sbp, rmse_sbp = eval_fold(model_sbp, test_loader, range_SBP, target_col=0)
        mae_dbp, rmse_dbp = eval_fold(model_dbp, test_loader, range_DBP, target_col=1)

        print(f'MAE SBP: {mae_sbp:.4f} mmHg | RMSE SBP: {rmse_sbp:.4f} mmHg')
        print(f'MAE DBP: {mae_dbp:.4f} mmHg | RMSE DBP: {rmse_dbp:.4f} mmHg')

        results['SBP']['maes'].append(mae_sbp)
        results['SBP']['rmses'].append(rmse_sbp)
        results['DBP']['maes'].append(mae_dbp)
        results['DBP']['rmses'].append(rmse_dbp)
    
    print(f"\n  Summary — {experiment_name}")
    for tag in ['SBP', 'DBP']:
        maes  = results[tag]['maes']
        rmses = results[tag]['rmses']
        print(f"  {tag} Mean MAE:  {np.mean(maes):.4f} ± {np.std(maes):.4f} mmHg")
        print(f"  {tag} Mean RMSE: {np.mean(rmses):.4f} mmHg")

    return results

In [1039]:
def decode_output_to_prediction(counts, value_range):
    """
    counts      : [n_output] tensor
    value_range : np.linspace(...)
    """
    rates = counts.numpy()
    total = rates.sum()

    if total > 0:
        rates_norm = rates / total
        sbp_pred = np.sum(rates_norm * value_range)
        sbp_std  = np.sqrt(np.sum(rates_norm * (value_range - sbp_pred) ** 2))
        return {
            'prediction': sbp_pred,
            'std': sbp_std,
            'rates': rates,
            'rates_norm': rates_norm,
            'values': value_range,
            'total_spikes': total
        }
    else:
        print('Could not make a prediction')
        return None


In [1040]:
def train_mlp_cv(X, Y, fold_indices, num_folds, epochs=100):

    results = []

    for fold in range(num_folds):

        test_idx = fold_indices[fold]
        train_idx = []
        for f in range(num_folds):
            if f != fold:
                train_idx.extend(fold_indices[f])

        X_train, X_test = X[train_idx].float(), X[test_idx].float()
        y_train, y_test = Y[train_idx], Y[test_idx]

        model = MLPRegressor(X.shape[1])
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()

        for epoch in range(epochs):
            model.train()
            optimizer.zero_grad()
            preds = model(X_train)
            loss = criterion(preds, y_train)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_test)
            mae = (preds - y_test).abs().mean().item()
            rmse = ((preds - y_test)**2).mean().sqrt().item()

        results.append((mae, rmse))

    maes = [r[0] for r in results]
    rmses = [r[1] for r in results]

    print("MLP RESULTS")
    print(f"MAE {np.mean(maes):.4f} ± {np.std(maes):.4f}")
    print(f"RMSE {np.mean(rmses):.4f}")

    return results

In [ ]:
spike_folder = './spiketrains'
spikes_SBP = torch.load(os.path.join(spike_folder, "RE_uniform_SBP"), weights_only=True)
combined_data, fold_indices = load_all_data_with_fold_indices()
Y_SBP = torch.tensor(combined_data['SP'].values, dtype=torch.float32)
Y_DBP = torch.tensor(combined_data['DP'].values, dtype=torch.float32)

config = {
    'n_input': 30,
    'n_hidden': 31,
    'n_output': 100,
    'beta': 0.95,
    'threshold': 0.9,
    'batch_size': 32,   # instead of feeding functions our dataset through sample by sample, we do it in batches
    'num_epochs': 30,
    'num_folds': 5,
    'lr': 5e-4
}

baseline_results = naive_baseline_cv(
    Y_SBP,
    Y_DBP,
    fold_indices,
    config['num_folds']
)

range_SBP = torch.linspace(70,190,100)
range_DBP = torch.linspace(45,100,100)

spike_trains = {'spikegen': 
                {'SBP': torch.load(os.path.join(spike_folder, "RE_spikegen_SBP"), weights_only=True), 
                 'DBP': torch.load(os.path.join(spike_folder, "RE_spikegen_DBP"), weights_only=True)},
                 'uniform': 
                {'SBP': torch.load(os.path.join(spike_folder, "RE_uniform_SBP"), weights_only=True), 
                 'DBP': torch.load(os.path.join(spike_folder, "RE_uniform_DBP"), weights_only=True)},
                 'beta': 
                {'SBP': torch.load(os.path.join(spike_folder, "RE_beta_SBP"), weights_only=True), 
                 'DBP': torch.load(os.path.join(spike_folder, "RE_beta_DBP"), weights_only=True)},
}

targets = {'SBP': (Y_SBP, range_SBP),
           'DBP': (Y_DBP, range_DBP)
}
'''
top_N_features_SBP = [
    'TNegSteepest_norm',
    'ppg_histogram_down_2',
    'vpg_max_0',
    'DWdivSW50',
    'ppg_min_0',
    'apg_b',
    'apg_min_0',
    'p2p_0',
    'SWaddDW50',
    'SW75_norm',
    'vpg_histogram_down_1',
    'apg_histogram_up_0',
    'S3_norm',
    'T_e_norm',
    'apg_histogram_down_2',
    'T_peak_e',
    'DW50_norm',
    'ppg4_histogram_down_5',
    'AUCsys',
    'apg_histogram_up_4',
    'ppg_b',
    'S2',
    'ppg4_histogram_down_2',
    'ppg3_histogram_down_9',
    'apg_max_0',
    'ppg_fft_peaks_heights_0',
    'vpg_histogram_up_2',
    'TSystoDiaRise_norm',
    'T_c',
    'hr_0'
]

top_N_features_DBP =  [
    'T_peak_a',
    'ppg4_histogram_down_2',
    'T_peak_e_norm',
    'ppg4_histogram_down_5',
    'vpg_histogram_down_8',
    'vpg_histogram_down_0',
    'ppg3_histogram_down_5',
    'DWdivSW75',
    'SW25',
    'T_peak_e',
    'vpg_histogram_down_7',
    'ppg4_histogram_up_2',
    'T_peak_c',
    'SW50_norm',
    'ppg4_histogram_down_0',
    'TNegSteepest',
    'apg_histogram_up_4',
    'ppg4_histogram_down_1',
    'AUCdia',
    'usdc_3',
    'ppg_e',
    'apg_histogram_up_3',
    'TSystoDiaRise_norm',
    'ppg3_max_0',
    'ppg4_histogram_up_0',
    'SW50',
    'ppg_histogram_down_5',
    'T_peak_a_norm',
    'SQI_kurtosis',
    'S4'
]


print("SBP mean:", Y_SBP.mean().item())
print("SBP std:", Y_SBP.std().item())
print("DBP mean:", Y_DBP.mean().item())
print("DBP std:", Y_DBP.std().item())

X_SBP = extract_and_normalize_top_features(combined_data, top_N_features_SBP)
X_DBP = extract_and_normalize_top_features(combined_data, top_N_features_DBP)

train_mlp_cv(X_SBP,Y_SBP,fold_indices,5,100)
train_mlp_cv(X_DBP,Y_DBP,fold_indices,5,100)
'''

all_results = {}

for encoding in spike_trains:
    for target in targets:
        spikes = spike_trains[encoding][target]
        all_results[encoding] = run_experiment(
            spikes, fold_indices, Y_SBP, Y_DBP,
            range_SBP, range_DBP, config,
            experiment_name=f"{encoding} encoding"
        )
print(f'{"="*50}')
print(f'FINAL RESULTS')
print(f'{"="*50}')
for encoding in all_results:
    for target in all_results[encoding]:
        res = all_results[encoding][target]
        maes = res['maes']
        rmses = res['rmses']
        print(f"  {encoding} - {target}: MAE {np.mean(maes):.4f} ± {np.std(maes):.4f} | RMSE {np.mean(rmses):.4f}")

Combined dataset shape: (3063, 222)
Total samples: 3063
Fold 0: 671 samples mapped to indices [0..670]
Fold 1: 614 samples mapped to indices [671..1284]
Fold 2: 652 samples mapped to indices [1285..1936]
Fold 3: 557 samples mapped to indices [1937..2493]
Fold 4: 569 samples mapped to indices [2494..3062]
NAIVE BASELINE RESULTS
SBP: MAE 11.6444 ± 3.1398 | RMSE 15.1859
DBP: MAE 7.6996 ± 0.8593 | RMSE 9.5675
SBP mean: 120.98946380615234
SBP std: 15.289740562438965
DBP mean: 67.22960662841797
DBP std: 9.304558753967285
MLP RESULTS
MAE 18.3370 ± 4.3269
RMSE 21.8450
MLP RESULTS
MAE 9.5794 ± 1.5129
RMSE 11.8978


'\nall_results = {}\n\nfor encoding in spike_trains:\n    for target in targets:\n        spikes = spike_trains[encoding][target]\n        all_results[encoding] = run_experiment(\n            spikes, fold_indices, Y_SBP, Y_DBP,\n            range_SBP, range_DBP, config,\n            experiment_name=f"{encoding} encoding"\n        )\nprint(f\'{"="*50}\')\nprint(f\'FINAL RESULTS\')\nprint(f\'{"="*50}\')\nfor encoding in all_results:\n    for target in all_results[encoding]:\n        res = all_results[encoding][target]\n        maes = res[\'maes\']\n        rmses = res[\'rmses\']\n        print(f"  {encoding} - {target}: MAE {np.mean(maes):.4f} ± {np.std(maes):.4f} | RMSE {np.mean(rmses):.4f}")\n'